# CIFAR100 flip helper

This notebook scans `results/cifar100/*.csv`, finds the rows whose `image` contains `label_55__idx_1524` or `label_7__idx_1605`, swaps `result`, `bab_time`, and `all_time` between the paired `fix_mask` and `fix_nonmask` rows, and writes flipped copies plus per-file logs.


In [1]:
from __future__ import annotations

from pathlib import Path
import re

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "results" / "cifar100").exists() and (candidate / "analysis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the XAIV repository root from the current working directory.")


def normalize_tag(tag):
    if pd.isna(tag):
        return tag
    value = str(tag).strip().lower().replace("-", "_").replace(" ", "_")
    mapping = {
        "fixmask": "fix_mask",
        "fix_mask": "fix_mask",
        "fixed_mask": "fix_mask",
        "nonmask": "fix_nonmask",
        "non_mask": "fix_nonmask",
        "fixnonmask": "fix_nonmask",
        "fix_nonmask": "fix_nonmask",
        "fixed_nonmask": "fix_nonmask",
    }
    return mapping.get(value, value)


LABEL_IDX_PATTERN = re.compile(r"label_(\d+)__idx_(\d+)")


def extract_label_idx(image_value):
    if pd.isna(image_value):
        return None
    match = LABEL_IDX_PATTERN.search(str(image_value))
    if not match:
        return None
    return int(match.group(1)), int(match.group(2))


def values_differ(left, right):
    if pd.isna(left) and pd.isna(right):
        return False
    return left != right


ROOT = find_repo_root()
CSV_DIR = ROOT / "results" / "cifar100"

TARGET_IMAGES = [
    {"label": 55, "idx": 1524},
    {"label": 7, "idx": 1605},
]
TARGET_KEYS = {(item["label"], item["idx"]) for item in TARGET_IMAGES}

PAIR_COLUMN_CANDIDATES = ["image", "k", "eps", "segment_index", "model", "onnx"]
SWAP_COLUMN_CANDIDATES = ["result", "bab_time", "all_time"]
RESULT_TAGS = ["fix_mask", "fix_nonmask"]

input_csvs = sorted(
    path
    for path in CSV_DIR.glob("*.csv")
    if "_flipped" not in path.stem and "_swap_log" not in path.stem and "_summary" not in path.stem
)

overall_summary = []
overall_log_frames = []
overall_changed_frames = []
overall_skipped_target_frames = []
skipped_files = []

for input_csv in input_csvs:
    try:
        df = pd.read_csv(input_csv).copy()
    except Exception as exc:
        skipped_files.append({
            "csv": input_csv.name,
            "reason": f"could not read CSV: {exc}",
        })
        continue

    missing_required = sorted({"image", "tag", "result"} - set(df.columns))
    if missing_required:
        skipped_files.append({
            "csv": input_csv.name,
            "reason": f"missing required columns: {', '.join(missing_required)}",
        })
        continue

    pair_columns = [column for column in PAIR_COLUMN_CANDIDATES if column in df.columns]
    if "image" not in pair_columns:
        skipped_files.append({
            "csv": input_csv.name,
            "reason": "expected an image column for pairing rows",
        })
        continue

    swap_columns = [column for column in SWAP_COLUMN_CANDIDATES if column in df.columns]
    if "result" not in swap_columns:
        skipped_files.append({
            "csv": input_csv.name,
            "reason": "expected a result column to swap",
        })
        continue

    output_csv = input_csv.with_name(f"{input_csv.stem}_flipped.csv")
    log_csv = input_csv.with_name(f"{input_csv.stem}_flipped_log.csv")

    df["tag_norm"] = df["tag"].apply(normalize_tag)
    df["label_idx"] = df["image"].apply(extract_label_idx)

    for column in swap_columns:
        df[f"old_{column}"] = df[column]

    target_rows = df[
        df["label_idx"].isin(TARGET_KEYS)
        & df["tag_norm"].isin(RESULT_TAGS)
    ].copy()

    fix_mask_rows = (
        target_rows[target_rows["tag_norm"] == "fix_mask"]
        .reset_index()
        .rename(columns={"index": "fix_mask_row_index"})
    )
    fix_nonmask_rows = (
        target_rows[target_rows["tag_norm"] == "fix_nonmask"]
        .reset_index()
        .rename(columns={"index": "fix_nonmask_row_index"})
    )

    change_log = []
    skipped_targets = []

    for target in TARGET_IMAGES:
        target_key = (target["label"], target["idx"])
        image_mask = fix_mask_rows[fix_mask_rows["label_idx"] == target_key].copy()
        image_nonmask = fix_nonmask_rows[fix_nonmask_rows["label_idx"] == target_key].copy()

        if image_mask.empty or image_nonmask.empty:
            skipped_targets.append({
                "csv": input_csv.name,
                "label": target["label"],
                "idx": target["idx"],
                "fix_mask_rows": len(image_mask),
                "fix_nonmask_rows": len(image_nonmask),
                "paired_rows": 0,
                "reason": "missing fix_mask or fix_nonmask rows",
            })
            continue

        try:
            paired_rows = image_mask.merge(
                image_nonmask,
                on=pair_columns,
                how="outer",
                suffixes=("_mask", "_nonmask"),
                indicator=True,
                validate="one_to_one",
            )
        except pd.errors.MergeError as exc:
            skipped_targets.append({
                "csv": input_csv.name,
                "label": target["label"],
                "idx": target["idx"],
                "fix_mask_rows": len(image_mask),
                "fix_nonmask_rows": len(image_nonmask),
                "paired_rows": 0,
                "reason": str(exc),
            })
            continue

        unmatched_rows = paired_rows[paired_rows["_merge"] != "both"].copy()
        paired_rows = paired_rows[paired_rows["_merge"] == "both"].copy()

        if not unmatched_rows.empty or paired_rows.empty:
            skipped_targets.append({
                "csv": input_csv.name,
                "label": target["label"],
                "idx": target["idx"],
                "fix_mask_rows": len(image_mask),
                "fix_nonmask_rows": len(image_nonmask),
                "paired_rows": len(paired_rows),
                "reason": "could not align fix_mask and fix_nonmask rows one-to-one",
            })
            continue

        for _, row in paired_rows.iterrows():
            mask_index = int(row["fix_mask_row_index"])
            nonmask_index = int(row["fix_nonmask_row_index"])

            log_entry = {
                "csv": input_csv.name,
                "output_csv": output_csv.name,
                "label": target["label"],
                "idx": target["idx"],
                "fix_mask_row_index": mask_index,
                "fix_nonmask_row_index": nonmask_index,
            }
            for column in pair_columns:
                log_entry[column] = row[column]

            pair_changed = False
            for column in swap_columns:
                old_mask_value = row[f"{column}_mask"]
                old_nonmask_value = row[f"{column}_nonmask"]

                df.at[mask_index, column] = old_nonmask_value
                df.at[nonmask_index, column] = old_mask_value

                log_entry[f"fix_mask_old_{column}"] = old_mask_value
                log_entry[f"fix_mask_new_{column}"] = old_nonmask_value
                log_entry[f"fix_nonmask_old_{column}"] = old_nonmask_value
                log_entry[f"fix_nonmask_new_{column}"] = old_mask_value

                column_changed = values_differ(old_mask_value, old_nonmask_value)
                log_entry[f"{column}_differed"] = column_changed
                pair_changed = pair_changed or column_changed

            log_entry["any_swapped_value_changed"] = pair_changed
            change_log.append(log_entry)

    changed_row_mask = pd.Series(False, index=df.index)
    for column in swap_columns:
        old_column = df[f"old_{column}"]
        new_column = df[column]
        changed_row_mask = changed_row_mask | (
            old_column.ne(new_column) & ~(old_column.isna() & new_column.isna())
        )

    changed_rows = df[
        df["label_idx"].isin(TARGET_KEYS)
        & df["tag_norm"].isin(RESULT_TAGS)
        & changed_row_mask
    ].copy()

    log_columns = ["csv", "output_csv", "label", "idx", *pair_columns, "fix_mask_row_index", "fix_nonmask_row_index"]
    for column in swap_columns:
        log_columns.extend([
            f"fix_mask_old_{column}",
            f"fix_mask_new_{column}",
            f"fix_nonmask_old_{column}",
            f"fix_nonmask_new_{column}",
            f"{column}_differed",
        ])
    log_columns.append("any_swapped_value_changed")

    change_log_df = pd.DataFrame(change_log, columns=log_columns)
    if not change_log_df.empty:
        sort_columns = [column for column in ["label", "idx", *pair_columns] if column in change_log_df.columns]
        change_log_df = change_log_df.sort_values(sort_columns).reset_index(drop=True)

    skipped_target_df = pd.DataFrame(skipped_targets)
    skipped_target_csv = f"{input_csv.stem}_flipped_skipped.csv"
    skipped_target_df.to_csv(input_csv.with_name(skipped_target_csv), index=False)
    if not skipped_target_df.empty:
        overall_skipped_target_frames.append(skipped_target_df)

    summary_row = {
        "csv": input_csv.name,
        "rows": len(df),
        "target_rows_found": len(target_rows),
        "pairs_swapped": len(change_log_df),
        "changed_pairs": int(change_log_df["any_swapped_value_changed"].sum()) if not change_log_df.empty else 0,
        "changed_rows": len(changed_rows),
        "swap_columns": ", ".join(swap_columns),
        "output_csv": output_csv.name,
        "log_csv": log_csv.name,
        "skipped_targets": len(skipped_target_df),
        "skipped_target_csv": skipped_target_csv,
    }
    overall_summary.append(summary_row)

    df.drop(
        columns=["tag_norm", "label_idx", *[f"old_{column}" for column in swap_columns]],
        errors="ignore",
    ).to_csv(output_csv, index=False)
    change_log_df.to_csv(log_csv, index=False)

    if not change_log_df.empty:
        overall_log_frames.append(change_log_df)
    if not changed_rows.empty:
        changed_rows["source_csv"] = input_csv.name
        overall_changed_frames.append(changed_rows)

summary_df = pd.DataFrame(overall_summary)
if not summary_df.empty:
    summary_df = summary_df.sort_values("csv").reset_index(drop=True)

combined_log_df = pd.concat(overall_log_frames, ignore_index=True) if overall_log_frames else pd.DataFrame()
combined_changed_df = pd.concat(overall_changed_frames, ignore_index=True) if overall_changed_frames else pd.DataFrame()
combined_skipped_targets_df = pd.concat(overall_skipped_target_frames, ignore_index=True) if overall_skipped_target_frames else pd.DataFrame()
skipped_files_df = pd.DataFrame(skipped_files)

summary_csv = CSV_DIR / "cifar100_flipped_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print(f"Repository root: {ROOT}")
print(f"Input directory: {CSV_DIR}")
print(f"Target label/idx pairs: {sorted(TARGET_KEYS)}")
print(f"Processed CSV files: {len(summary_df)}")
print(f"Skipped CSV files: {len(skipped_files_df)}")
print(f"Summary CSV: {summary_csv}")

display(summary_df)

if not skipped_files_df.empty:
    print("\nSkipped files:")
    display(skipped_files_df)

if not combined_skipped_targets_df.empty:
    print("\nTargets that were not paired cleanly:")
    display(combined_skipped_targets_df)

if not combined_changed_df.empty:
    changed_display_columns = [
        column
        for column in ["source_csv", "image", "k", "eps", "segment_index", "tag", "result", "old_result", "bab_time", "old_bab_time", "all_time", "old_all_time"]
        if column in combined_changed_df.columns
    ]
    reordered_columns = [
        column
        for column in ["source_csv", "image", "k", "eps", "segment_index", "tag", "old_result", "result", "old_bab_time", "bab_time", "old_all_time", "all_time"]
        if column in combined_changed_df.columns
    ]
    print("\nChanged rows:")
    display(
        combined_changed_df[reordered_columns or changed_display_columns]
        .sort_values([column for column in ["source_csv", "image", "tag"] if column in combined_changed_df.columns])
        .reset_index(drop=True)
    )

if not combined_log_df.empty:
    print("\nCombined swap log:")
    display(combined_log_df)


Repository root: /Users/zd3504phd/Desktop/XAIV
Input directory: /Users/zd3504phd/Desktop/XAIV/results/cifar100
Target label/idx pairs: [(7, 1605), (55, 1524)]
Processed CSV files: 5
Skipped CSV files: 1
Summary CSV: /Users/zd3504phd/Desktop/XAIV/results/cifar100/cifar100_flipped_summary.csv


,csv,rows,target_rows_found,pairs_swapped,changed_pairs,changed_rows,swap_columns,output_csv,log_csv,skipped_targets,skipped_target_csv
0,cifar100_CSI.csv,600,4,2,2,4,"result, bab_time, all_time",cifar100_CSI_flipped.csv,cifar100_CSI_flipped_log.csv,0,cifar100_CSI_flipped_skipped.csv
1,cifar100_CSI_d_1.csv,600,4,2,2,4,"result, bab_time, all_time",cifar100_CSI_d_1_flipped.csv,cifar100_CSI_d_1_flipped_log.csv,0,cifar100_CSI_d_1_flipped_skipped.csv
2,cifar100_CSI_d_2.csv,600,4,2,2,4,"result, bab_time, all_time",cifar100_CSI_d_2_flipped.csv,cifar100_CSI_d_2_flipped_log.csv,0,cifar100_CSI_d_2_flipped_skipped.csv
3,cifar100_CSI_d_3.csv,600,4,2,2,4,"result, bab_time, all_time",cifar100_CSI_d_3_flipped.csv,cifar100_CSI_d_3_flipped_log.csv,0,cifar100_CSI_d_3_flipped_skipped.csv
4,cifar100_CSI_d_4.csv,600,4,2,2,4,"result, bab_time, all_time",cifar100_CSI_d_4_flipped.csv,cifar100_CSI_d_4_flipped_log.csv,0,cifar100_CSI_d_4_flipped_skipped.csv



Skipped files:


,csv,reason
0,cifar100_abcrown.csv,could not read CSV: Error tokenizing data. C e...



Changed rows:


,source_csv,image,k,eps,segment_index,tag,old_result,result,old_bab_time,bab_time,old_all_time,all_time
0,cifar100_CSI.csv,resnet_large__label_55__idx_1524,1024,0.0039,0,fix_mask,unsat True,unsat False,NaN,0.155003,2.837473,11.863782
1,cifar100_CSI.csv,resnet_large__label_55__idx_1524,1024,0.0039,0,fix_nonmask,unsat False,unsat True,0.155003,NaN,11.863782,2.837473
2,cifar100_CSI.csv,resnet_medium__label_7__idx_1605,1024,0.0039,0,fix_mask,unsat True,unsat False,NaN,0.149080,3.635299,14.680127
3,cifar100_CSI.csv,resnet_medium__label_7__idx_1605,1024,0.0039,0,fix_nonmask,unsat False,unsat True,0.149080,NaN,14.680127,3.635299
4,cifar100_CSI_d_1.csv,resnet_large__label_55__idx_1524,1024,0.0039,0,fix_mask,unsat True,timeout False,NaN,0.157947,3.947368,101.584850
5,cifar100_CSI_d_1.csv,resnet_large__label_55__idx_1524,1024,0.0039,0,fix_nonmask,timeout False,unsat True,0.157947,NaN,101.584850,3.947368
6,cifar100_CSI_d_1.csv,resnet_medium__label_7__idx_1605,1024,0.0039,0,fix_mask,unsat True,unsat False,NaN,0.228751,2.472072,51.618666
7,cifar100_CSI_d_1.csv,resnet_medium__label_7__idx_1605,1024,0.0039,0,fix_nonmask,unsat False,unsat True,0.228751,NaN,51.618666,2.472072
8,cifar100_CSI_d_2.csv,resnet_large__label_55__idx_1524,1024,0.0039,0,fix_mask,unsat True,timeout False,NaN,0.289007,2.873369,103.132182
9,cifar100_CSI_d_2.csv,resnet_large__label_55__idx_1524,1024,0.0039,0,fix_nonmask,timeout False,unsat True,0.289007,NaN,103.132182,2.873369



Combined swap log:


,csv,output_csv,label,idx,image,k,eps,segment_index,model,onnx,...,fix_mask_new_bab_time,fix_nonmask_old_bab_time,fix_nonmask_new_bab_time,bab_time_differed,fix_mask_old_all_time,fix_mask_new_all_time,fix_nonmask_old_all_time,fix_nonmask_new_all_time,all_time_differed,any_swapped_value_changed
0,cifar100_CSI.csv,cifar100_CSI_flipped.csv,7,1605,resnet_medium__label_7__idx_1605,1024,0.0039,0,CIFAR100_resnet_medium,onnx/CIFAR100_resnet_medium.onnx,...,0.149080,0.149080,NaN,True,3.635299,14.680127,14.680127,3.635299,True,True
1,cifar100_CSI.csv,cifar100_CSI_flipped.csv,55,1524,resnet_large__label_55__idx_1524,1024,0.0039,0,CIFAR100_resnet_large,onnx/CIFAR100_resnet_large.onnx,...,0.155003,0.155003,NaN,True,2.837473,11.863782,11.863782,2.837473,True,True
2,cifar100_CSI_d_1.csv,cifar100_CSI_d_1_flipped.csv,7,1605,resnet_medium__label_7__idx_1605,1024,0.0039,0,CIFAR100_resnet_medium,onnx/CIFAR100_resnet_medium.onnx,...,0.228751,0.228751,NaN,True,2.472072,51.618666,51.618666,2.472072,True,True
3,cifar100_CSI_d_1.csv,cifar100_CSI_d_1_flipped.csv,55,1524,resnet_large__label_55__idx_1524,1024,0.0039,0,CIFAR100_resnet_large,onnx/CIFAR100_resnet_large.onnx,...,0.157947,0.157947,NaN,True,3.947368,101.584850,101.584850,3.947368,True,True
4,cifar100_CSI_d_2.csv,cifar100_CSI_d_2_flipped.csv,7,1605,resnet_medium__label_7__idx_1605,1024,0.0039,0,CIFAR100_resnet_medium,onnx/CIFAR100_resnet_medium.onnx,...,0.146204,0.146204,NaN,True,2.601264,102.126904,102.126904,2.601264,True,True
5,cifar100_CSI_d_2.csv,cifar100_CSI_d_2_flipped.csv,55,1524,resnet_large__label_55__idx_1524,1024,0.0039,0,CIFAR100_resnet_large,onnx/CIFAR100_resnet_large.onnx,...,0.289007,0.289007,NaN,True,2.873369,103.132182,103.132182,2.873369,True,True
6,cifar100_CSI_d_3.csv,cifar100_CSI_d_3_flipped.csv,7,1605,resnet_medium__label_7__idx_1605,1024,0.0039,0,CIFAR100_resnet_medium,onnx/CIFAR100_resnet_medium.onnx,...,0.168976,0.168976,NaN,True,2.446434,102.815084,102.815084,2.446434,True,True
7,cifar100_CSI_d_3.csv,cifar100_CSI_d_3_flipped.csv,55,1524,resnet_large__label_55__idx_1524,1024,0.0039,0,CIFAR100_resnet_large,onnx/CIFAR100_resnet_large.onnx,...,0.236031,0.236031,NaN,True,3.462038,104.747899,104.747899,3.462038,True,True
8,cifar100_CSI_d_4.csv,cifar100_CSI_d_4_flipped.csv,7,1605,resnet_medium__label_7__idx_1605,1024,0.0039,0,CIFAR100_resnet_medium,onnx/CIFAR100_resnet_medium.onnx,...,0.187193,0.187193,NaN,True,3.031911,107.346983,107.346983,3.031911,True,True
9,cifar100_CSI_d_4.csv,cifar100_CSI_d_4_flipped.csv,55,1524,resnet_large__label_55__idx_1524,1024,0.0039,0,CIFAR100_resnet_large,onnx/CIFAR100_resnet_large.onnx,...,0.148917,0.148917,NaN,True,3.296631,106.864831,106.864831,3.296631,True,True
